# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 8: Advanced LLM Agents</font>

# <font color="#003660">LLM Agent Plannung - ToDo Lists</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... will know how LLM agents are build with LangChain and LangGraph. <br>
        ... will know how states in LLM agents work. <br>
        ... will know how states can be used for ToDo Lists as planning and progress indicator in LLM agents. <br>
    </font>
</div>
</p>

The following content is heavily inspired by the following excellent sources:

* [Langchain-AI Deep Agents from Scratch](https://github.com/langchain-ai/deep-agents-from-scratch)
* [LangChain Academy](https://academy.langchain.com/)
* [Introduction to LangChain Agents](https://github.com/langchain-ai/langchain-academy/blob/main/module-1/agent.ipynb)
* [LangChain Docs (Python)](https://python.langchain.com/)

In [ ]:
!pip install -U wikipedia langchain langchain-community langchain-openai deepagents

Today we will setup our own ollama server. We can do this directly in Google Colab.

First we need to install the pciutil package (to let ollama automatically detect GPU) and ollama. Just run the code below.

In [ ]:
# with this linux package, ollama can then detect GPU, if available
!sudo apt-get install -y pciutils

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

The next chunk is to start ollama locally as a subprocess in the background. (Even if ollama tells you that it has started the server, it has not.)

In [ ]:
import subprocess

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

Now we will have to download both models for this session. Run the code below.

In [ ]:
!ollama pull gpt-oss # takes around 2 minutes

## Offload Context using Middleware

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from deepagents.middleware.filesystem import FilesystemMiddleware

config = {
    "model": "gpt-oss",
    "base_url": "http://127.0.0.1:11434/v1",
	"api_key": "ollama",
    "max_tokens": 8192,
    "temperature": 0,
    "seed": 42,
}

model=ChatOpenAI(
    **config
)

# FilesystemMiddleware is included by default in create_deep_agent
# You can customize it if building a custom agent
agent = create_agent(
    model=model,
    # TODO: add the filesystem here
)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "Write me a python script that prints 'Hello, World!' to the console and saves it to a file named hello.py in the directory /content/"}]})

print(result.keys())

In [ ]:
# Print the agent's response
for m in result["messages"]:
    m.pretty_print()

## Build your own Deep Agent (Blueprint)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent
from langchain_community.retrievers import WikipediaRetriever

research_instructions = """\
You are an expert researcher.
When getting a research task follow this procedure:
1. Write a to do list using the write_todo tool.
2. Follow your todo list.
3. Write a report of your research in a file using write_file tool.
"""

def search_in_wikipedia(query: str) -> str:
    """Search Wikipedia for a given query."""
    retriever = WikipediaRetriever(load_max_docs=1)
    docs = retriever.invoke(query)
    results = "\n\n-----\n\n".join([f"Document {i}:\n\nMetadata:\n-Title: {docs[i].metadata['title'].strip()}\n-Path: https://en.wikipedia.org/wiki/{docs[i].metadata['title'].strip().replace(' ', '_')}\n\nContent:\n{docs[i].metadata['summary'].strip()}" for i in range(len(docs))])

    return results

model=ChatOpenAI(
    **config
)

agent = create_deep_agent(
    model=model,
    system_prompt=research_instructions,
    tools=[search_in_wikipedia],
)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "1. Conduct research on the impact of the relationship of Jon Snow and Daenerys Targaryen using Wikipedia sources. 2. Write a 1000 word report with citations. 3. Write the report as a markdown file named jon_snow_daenerys_relationship.md using the write_file tool."}]})

In [ ]:
# Print the agent's response
for m in result["messages"]:
    print(m.pretty_repr())

In [ ]:
import json
print(json.dumps(result["files"], indent=4))

IMPORTANT: This file gets lost when re-running the agent.

## How to store persistently across state?

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()

def make_backend(runtime):
    return CompositeBackend(
        default=StateBackend(runtime),  # Ephemeral storage
        routes={
            "/memories/": StoreBackend(runtime)  # Persistent storage
        }
    )

agent = create_deep_agent(
    model=model,
    system_prompt=research_instructions,
    tools=[search_in_wikipedia],
    store=InMemoryStore(),  # Required for StoreBackend
    backend=make_backend,
    checkpointer=checkpointer
)

Adding this uuid with a config you can transfer the files across threads.

In [ ]:
import uuid
config1 = {"configurable": {"thread_id": str(uuid.uuid4())}}
result = agent.invoke({"messages": [{"role": "user", "content": "1. Conduct research on the impact of the relationship of Jon Snow and Daenerys Targaryen using the search_in_wikipedia tool. 2. Write a 1000 word report with citations. 3. Write the report as a **persistent** markdown file with the explicit file path **/memories/jon_snow_daenerys_relationship.md** using the write_file tool."}]}, config=config1)

In [ ]:
# Print the agent's response
for m in result["messages"]:
    print(m.pretty_repr())